# Build Master Monthly Dataset

This notebook merges:
- cleaned monthly WARN layoffs
- FRED macroeconomic indicators
- GDELT monthly news features

Final output:
- ../data/processed/master_monthly.csv

In [1]:
import pandas as pd
import numpy as np
import os

In [7]:
def load_fred_series(path, new_name):
    s = pd.read_csv(path)
    s.columns = [str(c).strip() for c in s.columns]

    # Handle either DATE or observation_date
    if "DATE" in s.columns:
        date_col = "DATE"
    elif "observation_date" in s.columns:
        date_col = "observation_date"
    else:
        raise ValueError(f"No recognized date column found in {path}. Columns: {list(s.columns)}")

    value_cols = [c for c in s.columns if c != date_col]
    if len(value_cols) != 1:
        raise ValueError(f"Expected 1 value column in {path}, got {value_cols}")

    value_col = value_cols[0]

    s[date_col] = pd.to_datetime(s[date_col], errors="coerce")
    s[value_col] = pd.to_numeric(s[value_col].replace(".", pd.NA), errors="coerce")
    s = s.dropna(subset=[date_col, value_col]).copy()

    s["month"] = s[date_col].dt.to_period("M").dt.to_timestamp()

    s_monthly = (
        s.groupby("month", as_index=False)[value_col]
        .mean()
        .rename(columns={value_col: new_name})
    )

    return s_monthly

In [8]:
# Load cleaned WARN target
warn_monthly = pd.read_csv("../data/processed/warn_monthly.csv")
warn_monthly["month"] = pd.to_datetime(warn_monthly["month"])

# Load FRED
caurn = load_fred_series("../data/raw/fred/CAURN.csv", "ca_unemployment_rate")
fedfunds = load_fred_series("../data/raw/fred/FEDFUNDS.csv", "fed_funds_rate")
indeed = load_fred_series("../data/raw/fred/IHLIDXUSCA.csv", "indeed_job_postings_index")

# Load GDELT
gdelt = pd.read_csv("../data/raw/gdelt/gdelt_news_monthly.csv")
gdelt["month"] = pd.to_datetime(gdelt["month"])
gdelt["news_volume"] = pd.to_numeric(gdelt["news_volume"], errors="coerce")
gdelt["news_tone"] = pd.to_numeric(gdelt["news_tone"], errors="coerce")

In [9]:
# Fixed month index for the exact project window
month_index = pd.DataFrame({
    "month": pd.date_range("2020-07-01", "2025-06-01", freq="MS")
})

master = (
    month_index
    .merge(warn_monthly, on="month", how="left")
    .merge(caurn, on="month", how="left")
    .merge(fedfunds, on="month", how="left")
    .merge(indeed, on="month", how="left")
    .merge(gdelt, on="month", how="left")
    .sort_values("month")
    .reset_index(drop=True)
)

# WARN is the target, so missing means zero layoffs recorded for that month
master["warn_layoffs"] = master["warn_layoffs"].fillna(0)

master.head()

,month,warn_layoffs,ca_unemployment_rate,fed_funds_rate,indeed_job_postings_index,news_volume,news_tone
0,2020-07-01,32814,NaN,NaN,NaN,64201,-2.080829
1,2020-08-01,15544,NaN,NaN,NaN,47418,-2.014797
2,2020-09-01,14535,NaN,NaN,NaN,41715,-1.978027
3,2020-10-01,10477,NaN,NaN,NaN,27663,-1.937167
4,2020-11-01,9323,NaN,NaN,NaN,19858,-1.985260


In [10]:
# Feature engineering
master["warn_layoffs_log1p"] = np.log1p(master["warn_layoffs"])
master["news_volume_log1p"] = np.log1p(master["news_volume"])

# 1-3 month lags for predictors
predictor_cols = [
    "ca_unemployment_rate",
    "fed_funds_rate",
    "indeed_job_postings_index",
    "news_volume",
    "news_volume_log1p",
    "news_tone"
]

for col in predictor_cols:
    for lag in [1, 2, 3]:
        master[f"{col}_lag{lag}"] = master[col].shift(lag)

# optional z-scores for overlay plots
plot_cols = [
    "warn_layoffs",
    "ca_unemployment_rate",
    "fed_funds_rate",
    "indeed_job_postings_index",
    "news_volume",
    "news_tone"
]

for col in plot_cols:
    std = master[col].std()
    if pd.notna(std) and std != 0:
        master[f"{col}_z"] = (master[col] - master[col].mean()) / std
    else:
        master[f"{col}_z"] = np.nan

master.head()

,month,warn_layoffs,ca_unemployment_rate,fed_funds_rate,indeed_job_postings_index,news_volume,news_tone,warn_layoffs_log1p,news_volume_log1p,ca_unemployment_rate_lag1,...,news_volume_log1p_lag3,news_tone_lag1,news_tone_lag2,news_tone_lag3,warn_layoffs_z,ca_unemployment_rate_z,fed_funds_rate_z,indeed_job_postings_index_z,news_volume_z,news_tone_z
0,2020-07-01,32814,NaN,NaN,NaN,64201,-2.080829,10.398641,11.069790,NaN,...,NaN,NaN,NaN,NaN,5.268537,NaN,NaN,NaN,4.414589,-1.912911
1,2020-08-01,15544,NaN,NaN,NaN,47418,-2.014797,9.651494,10.766778,NaN,...,NaN,-2.080829,NaN,NaN,1.767228,NaN,NaN,NaN,2.807788,-1.633807
2,2020-09-01,14535,NaN,NaN,NaN,41715,-1.978027,9.584384,10.638640,NaN,...,NaN,-2.014797,-2.080829,NaN,1.562664,NaN,NaN,NaN,2.261784,-1.478387
3,2020-10-01,10477,NaN,NaN,NaN,27663,-1.937167,9.257033,10.227887,NaN,...,11.069790,-1.978027,-2.014797,-2.080829,0.739948,NaN,NaN,NaN,0.916448,-1.305681
4,2020-11-01,9323,NaN,NaN,NaN,19858,-1.985260,9.140347,9.896413,NaN,...,10.766778,-1.937167,-1.978027,-2.014797,0.505987,NaN,NaN,NaN,0.169199,-1.508961


In [11]:
print(caurn.head())
print(fedfunds.head())
print(indeed.head())

       month  ca_unemployment_rate
0 2020-12-01                   8.9
1 2021-01-01                   9.1
2 2021-02-01                   8.8
3 2021-03-01                   8.5
4 2021-04-01                   8.3
       month  fed_funds_rate
0 2021-01-01            0.09
1 2021-02-01            0.08
2 2021-03-01            0.07
3 2021-04-01            0.07
4 2021-05-01            0.06
       month  indeed_job_postings_index
0 2021-01-01                  93.000000
1 2021-02-01                  96.236786
2 2021-03-01                 103.845484
3 2021-04-01                 115.979333
4 2021-05-01                 124.212903


In [13]:
master.shape

(60, 33)

In [15]:
master.isna().sum()

month                             0
warn_layoffs                      0
ca_unemployment_rate              5
fed_funds_rate                    6
indeed_job_postings_index         6
news_volume                       0
news_tone                         0
warn_layoffs_log1p                0
news_volume_log1p                 0
ca_unemployment_rate_lag1         6
ca_unemployment_rate_lag2         7
ca_unemployment_rate_lag3         8
fed_funds_rate_lag1               7
fed_funds_rate_lag2               8
fed_funds_rate_lag3               9
indeed_job_postings_index_lag1    7
indeed_job_postings_index_lag2    8
indeed_job_postings_index_lag3    9
news_volume_lag1                  1
news_volume_lag2                  2
news_volume_lag3                  3
news_volume_log1p_lag1            1
news_volume_log1p_lag2            2
news_volume_log1p_lag3            3
news_tone_lag1                    1
news_tone_lag2                    2
news_tone_lag3                    3
warn_layoffs_z              

In [12]:
os.makedirs("../data/processed", exist_ok=True)
master.to_csv("../data/processed/master_monthly.csv", index=False)

print("Saved: ../data/processed/master_monthly.csv")
print(master.shape)
master.isna().sum()

Saved: ../data/processed/master_monthly.csv
(60, 33)


month                             0
warn_layoffs                      0
ca_unemployment_rate              5
fed_funds_rate                    6
indeed_job_postings_index         6
news_volume                       0
news_tone                         0
warn_layoffs_log1p                0
news_volume_log1p                 0
ca_unemployment_rate_lag1         6
ca_unemployment_rate_lag2         7
ca_unemployment_rate_lag3         8
fed_funds_rate_lag1               7
fed_funds_rate_lag2               8
fed_funds_rate_lag3               9
indeed_job_postings_index_lag1    7
indeed_job_postings_index_lag2    8
indeed_job_postings_index_lag3    9
news_volume_lag1                  1
news_volume_lag2                  2
news_volume_lag3                  3
news_volume_log1p_lag1            1
news_volume_log1p_lag2            2
news_volume_log1p_lag3            3
news_tone_lag1                    1
news_tone_lag2                    2
news_tone_lag3                    3
warn_layoffs_z              